# 🏥 Clinical NLP RAG Pipeline
## Biomedical NER + Semantic Chunking + BGE Embeddings + FAISS + OpenBioLLM-8B

**Cotiviti Intern Assessment — Topic 1: Clinical Natural Language Technology**

**Author:** Arshad Naguru | MS Artificial Intelligence, Rochester Institute of Technology

---

### Full Pipeline Architecture
```
Full Patient Record (multiple notes + labs + reports)
          ↓
  Semantic Chunking (256 tokens, 64 overlap)
          ↓
  Biomedical Embeddings (BAAI/bge-large-en-v1.5)
          ↓
  FAISS Vector Index
          ↓
  Query → Retrieve top-k relevant chunks
          ↓
  NER on retrieved chunks (d4data/biomedical-ner-all)
          ↓
  OpenBioLLM-8B RAG-grounded generation
          ↓
  SOAP Note + ICD-10 codes + Entity Table
```

**GPU:** A100 40GB | **Expected VRAM usage:** ~8GB | **Free headroom:** ~32GB

## Cell 1 — Install Dependencies

In [5]:
!pip install -q transformers==4.45.0
!pip install -q accelerate==0.34.0
!pip install -q bitsandbytes==0.43.1
!pip install -q sentence-transformers==2.7.0
!pip install -q faiss-gpu
!pip install -q sentencepiece pandas numpy nltk

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 136.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 119.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.3/324.3 kB 27.0 MB/s eta 0:00:00


## Cell 2 — GPU Check & Configuration

In [2]:
import torch
import numpy as np

assert torch.cuda.is_available(), 'Please enable GPU: Runtime > Change runtime type > A100'

gpu = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'✅ GPU: {gpu}')
print(f'✅ VRAM: {vram:.1f} GB')
print(f'✅ System RAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB GPU')

LOAD_IN_4BIT = vram < 35
DEVICE = 'cuda'
print(f'\n📋 Config:')
print(f'   LLM mode: {"4-bit quantization" if LOAD_IN_4BIT else "bfloat16 full precision"}')
print(f'   Device: {DEVICE}')
print(f'   Expected VRAM usage: ~8GB (NER + BGE + LLM + FAISS)')
print(f'   Available headroom: ~{vram-8:.0f}GB')

✅ GPU: NVIDIA A100-SXM4-40GB
✅ VRAM: 42.4 GB
✅ System RAM: 42.4 GB GPU

📋 Config:
   LLM mode: bfloat16 full precision
   Device: cuda
   Expected VRAM usage: ~8GB (NER + BGE + LLM + FAISS)
   Available headroom: ~34GB


## Cell 3 — Simulated Full Patient Record

In production Cotiviti would ingest full claim packages — multiple clinical notes,
lab reports, discharge summaries, radiology reports. We simulate that here with
a realistic multi-document patient record for the same patient across 4 visits.

In [3]:
PATIENT_RECORD = {
    'Visit 1 - Initial Presentation': """
    Patient: John M., 58-year-old male. Date: January 15, 2024.
    Chief Complaint: Increased thirst, frequent urination, fatigue for 3 months.
    History: No prior diagnosis of diabetes. Family history significant for Type 2 diabetes
    mellitus in mother and brother. BMI 31.2, obese. Sedentary lifestyle.
    Vitals: BP 152/94 mmHg, HR 82, Temp 98.6F, SpO2 98%.
    Labs: Fasting glucose 218 mg/dL, HbA1c 9.4%, Total cholesterol 224 mg/dL,
    LDL 148 mg/dL, HDL 38 mg/dL, Triglycerides 210 mg/dL.
    Creatinine 1.3 mg/dL, eGFR 58, Urine albumin-creatinine ratio 42 mg/g.
    Assessment: New diagnosis Type 2 diabetes mellitus with hypertension and dyslipidemia.
    Plan: Start Metformin 500mg twice daily, titrate to 1000mg over 2 weeks.
    Start Lisinopril 10mg daily for hypertension and renal protection.
    Dietary counseling, exercise program. Follow-up in 6 weeks.
    """,

    'Visit 2 - Follow-up': """
    Patient: John M., 58-year-old male. Date: March 2, 2024.
    Interval History: Patient reports improved energy but persistent polyuria.
    Adherent to Metformin 1000mg twice daily. Lisinopril 10mg daily continued.
    Reports occasional mild nausea with Metformin, taking with meals.
    Vitals: BP 138/88 mmHg, HR 78, Weight 94.2kg (down 1.8kg).
    Labs: Fasting glucose 186 mg/dL, HbA1c 8.6%, Creatinine 1.4 mg/dL, eGFR 52.
    Potassium 4.2 mEq/L. LFTs within normal limits.
    Peripheral neuropathy examination: reduced sensation in bilateral lower extremities,
    monofilament test abnormal bilaterally.
    Assessment: Improving but suboptimal glycemic control. Early diabetic peripheral
    neuropathy. Mild chronic kidney disease stage 3a.
    Plan: Add Empagliflozin 10mg daily for additional glycemic control and renal
    protection. Refer to endocrinology and nephrology. Start Gabapentin 300mg
    nightly for neuropathic pain. Ophthalmology referral for diabetic eye exam.
    """,

    'Radiology Report - Chest XRay': """
    Patient: John M., DOB: 03/15/1966. Date: March 2, 2024.
    Examination: PA and lateral chest radiograph.
    Clinical indication: Hypertension, diabetes mellitus follow-up.
    Findings: Heart size is mildly enlarged with cardiothoracic ratio of 0.52.
    Mild pulmonary vascular congestion noted bilaterally. No acute consolidation,
    pleural effusion, or pneumothorax. Bony structures intact. Aortic knuckle
    is prominent consistent with atherosclerotic changes.
    Impression: Mild cardiomegaly with early pulmonary vascular redistribution.
    Atherosclerotic aortic changes. Clinical correlation recommended.
    Radiologist: Dr. Sarah Chen, MD. Signed: March 2, 2024 at 15:42.
    """,

    'Visit 3 - Endocrinology Consult': """
    Patient: John M., 58-year-old male. Date: April 18, 2024. Referring: Primary Care.
    Reason for referral: Poorly controlled Type 2 diabetes, consider medication adjustment.
    Current medications: Metformin 1000mg BID, Empagliflozin 10mg daily,
    Lisinopril 10mg daily, Gabapentin 300mg nightly.
    Allergies: Sulfa drugs - rash.
    Exam: Alert, oriented. No acute distress. Acanthosis nigricans noted on neck.
    Foot exam: Diminished sensation to monofilament bilateral feet. No ulcers.
    Labs today: HbA1c 8.1%, Fasting glucose 174 mg/dL, C-peptide 1.8 ng/mL.
    Insulin-to-glucose ratio suggests significant insulin resistance.
    Assessment: Type 2 DM with inadequate control on dual oral therapy.
    Diabetic peripheral neuropathy, CKD stage 3a, hypertension, dyslipidemia.
    Plan: Add Semaglutide 0.25mg weekly subcutaneous injection, titrate to 1mg.
    Increase Empagliflozin to 25mg daily. Start Atorvastatin 40mg for dyslipidemia.
    Continue Metformin, Lisinopril. Continuous glucose monitoring discussed.
    Diabetic educator referral. Recheck HbA1c in 3 months.
    """,

    'Lab Report - Comprehensive Panel': """
    Patient: John M. Date: April 18, 2024. Ordered by: Endocrinology.
    METABOLIC PANEL:
    Glucose: 174 mg/dL (H) [Reference: 70-100]
    BUN: 22 mg/dL [Reference: 7-25]
    Creatinine: 1.4 mg/dL (H) [Reference: 0.7-1.3]
    eGFR: 52 mL/min/1.73m2 (L) [Reference: >60]
    Sodium: 138 mEq/L [Reference: 136-145]
    Potassium: 4.1 mEq/L [Reference: 3.5-5.1]
    LIPID PANEL:
    Total Cholesterol: 218 mg/dL (H) [Reference: <200]
    LDL: 142 mg/dL (H) [Reference: <100]
    HDL: 39 mg/dL (L) [Reference: >40]
    Triglycerides: 198 mg/dL (H) [Reference: <150]
    DIABETES:
    HbA1c: 8.1% (H) [Reference: <5.7%]
    C-peptide: 1.8 ng/mL [Reference: 0.5-2.0]
    Urine ACR: 48 mg/g (H) [Reference: <30]
    LIVER FUNCTION:
    ALT: 28 U/L [Reference: 7-56]
    AST: 24 U/L [Reference: 10-40]
    All within normal limits.
    """
}

print(f'✅ Patient record loaded: {len(PATIENT_RECORD)} documents')
for name, text in PATIENT_RECORD.items():
    word_count = len(text.split())
    print(f'   📄 {name}: {word_count} words')

total_words = sum(len(t.split()) for t in PATIENT_RECORD.values())
print(f'\n   Total: {total_words} words across {len(PATIENT_RECORD)} documents')
print(f'   This would exceed context window without chunking — RAG pipeline needed')

✅ Patient record loaded: 5 documents
   📄 Visit 1 - Initial Presentation: 122 words
   📄 Visit 2 - Follow-up: 125 words
   📄 Radiology Report - Chest XRay: 81 words
   📄 Visit 3 - Endocrinology Consult: 134 words
   📄 Lab Report - Comprehensive Panel: 106 words

   Total: 568 words across 5 documents
   This would exceed context window without chunking — RAG pipeline needed


## Cell 4 — Semantic Chunking with Overlap

Split each document into semantically coherent chunks rather than arbitrary
character splits. Uses sentence boundaries + sliding window with 64-token overlap
to preserve context across chunk boundaries.

In [4]:
import nltk
from nltk.tokenize import sent_tokenize
from typing import List, Dict

def count_tokens_approx(text: str) -> int:
    """Approximate token count (words * 1.3 is close for biomedical text)."""
    return int(len(text.split()) * 1.3)

def semantic_chunk(
    text: str,
    doc_name: str,
    chunk_size: int = 256,
    overlap_tokens: int = 64
) -> List[Dict]:
    """
    Semantically chunk a clinical document.
    - Split into sentences first (respects clinical sentence boundaries)
    - Group sentences into chunks of ~chunk_size tokens
    - Add overlap_tokens of context from previous chunk
    """
    sentences = sent_tokenize(text.strip())
    sentences = [s.strip() for s in sentences if len(s.strip()) > 10]

    chunks = []
    current_chunk_sents = []
    current_tokens = 0
    overlap_buffer = []  # sentences kept for overlap

    for sent in sentences:
        sent_tokens = count_tokens_approx(sent)

        if current_tokens + sent_tokens > chunk_size and current_chunk_sents:
            # Save current chunk
            chunk_text = ' '.join(current_chunk_sents)
            chunks.append({
                'text': chunk_text,
                'doc': doc_name,
                'chunk_id': len(chunks),
                'tokens': current_tokens
            })

            # Build overlap: keep last N tokens worth of sentences
            overlap_sents = []
            overlap_count = 0
            for s in reversed(current_chunk_sents):
                t = count_tokens_approx(s)
                if overlap_count + t <= overlap_tokens:
                    overlap_sents.insert(0, s)
                    overlap_count += t
                else:
                    break

            # Start new chunk with overlap
            current_chunk_sents = overlap_sents + [sent]
            current_tokens = overlap_count + sent_tokens
        else:
            current_chunk_sents.append(sent)
            current_tokens += sent_tokens

    # Don't forget last chunk
    if current_chunk_sents:
        chunk_text = ' '.join(current_chunk_sents)
        chunks.append({
            'text': chunk_text,
            'doc': doc_name,
            'chunk_id': len(chunks),
            'tokens': current_tokens
        })

    return chunks


# Chunk all documents in the patient record
all_chunks = []
print('Semantic chunking patient record...')
print('='*60)

for doc_name, doc_text in PATIENT_RECORD.items():
    doc_chunks = semantic_chunk(doc_text, doc_name, chunk_size=256, overlap_tokens=64)
    all_chunks.extend(doc_chunks)
    print(f'📄 {doc_name}')
    print(f'   → {len(doc_chunks)} chunks')
    for c in doc_chunks:
        print(f'     Chunk {c["chunk_id"]}: {c["tokens"]} tokens | {c["text"][:80]}...')
    print()

print(f'✅ Total chunks: {len(all_chunks)}')
print(f'   Avg tokens per chunk: {sum(c["tokens"] for c in all_chunks)/len(all_chunks):.0f}')

Semantic chunking patient record...
📄 Visit 1 - Initial Presentation
   → 1 chunks
     Chunk 0: 153 tokens | Patient: John M., 58-year-old male. Date: January 15, 2024. Chief Complaint: Inc...

📄 Visit 2 - Follow-up
   → 1 chunks
     Chunk 0: 153 tokens | Patient: John M., 58-year-old male. Date: March 2, 2024. Interval History: Patie...

📄 Radiology Report - Chest XRay
   → 1 chunks
     Chunk 0: 97 tokens | Patient: John M., DOB: 03/15/1966. Date: March 2, 2024. Examination: PA and late...

📄 Visit 3 - Endocrinology Consult
   → 1 chunks
     Chunk 0: 161 tokens | Patient: John M., 58-year-old male. Date: April 18, 2024. Referring: Primary Car...

📄 Lab Report - Comprehensive Panel
   → 1 chunks
     Chunk 0: 136 tokens | Patient: John M. Date: April 18, 2024. Ordered by: Endocrinology. METABOLIC PANE...

✅ Total chunks: 5
   Avg tokens per chunk: 140


## Cell 5 — Load BGE-Large Embeddings + Build FAISS Index

`BAAI/bge-large-en-v1.5` — one of the strongest general embedding models.
1024-dimensional dense embeddings. Fast on GPU. Excellent semantic search quality.

FAISS (Facebook AI Similarity Search) — in-memory vector index for instant retrieval.

In [6]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

print('Loading BGE-Large embedding model...')
embedder = SentenceTransformer('BAAI/bge-large-en-v1.5', device=DEVICE)
print(f'✅ BGE-Large loaded | Embedding dim: {embedder.get_sentence_embedding_dimension()}')

# Embed all chunks
print('\nEmbedding all chunks...')
chunk_texts = [c['text'] for c in all_chunks]

# BGE models benefit from a query prefix for retrieval
# For documents (corpus side), no prefix needed
corpus_embeddings = embedder.encode(
    chunk_texts,
    batch_size=16,
    show_progress_bar=True,
    normalize_embeddings=True  # Important for cosine similarity with FAISS IP
)

print(f'\n✅ Embeddings shape: {corpus_embeddings.shape}')
print(f'   {len(all_chunks)} chunks x {corpus_embeddings.shape[1]} dimensions')

# Build FAISS index (Inner Product = cosine similarity with normalized vectors)
print('\nBuilding FAISS index...')
embedding_dim = corpus_embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(embedding_dim)

# Move to GPU for faster search
res = faiss.StandardGpuResources()
faiss_index_gpu = faiss.index_cpu_to_gpu(res, 0, faiss_index)
faiss_index_gpu.add(corpus_embeddings.astype(np.float32))

print(f'✅ FAISS GPU index built')
print(f'   Vectors indexed: {faiss_index_gpu.ntotal}')
print(f'   Index type: Flat Inner Product (cosine similarity)')
print(f'   Search: exact nearest neighbor — no approximation')

# Memory report
allocated = torch.cuda.memory_allocated() / 1e9
print(f'\n📊 VRAM used so far: {allocated:.2f} GB / 40.0 GB')

Loading BGE-Large embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

✅ BGE-Large loaded | Embedding dim: 1024

Embedding all chunks...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


✅ Embeddings shape: (5, 1024)
   5 chunks x 1024 dimensions

Building FAISS index...
✅ FAISS GPU index built
   Vectors indexed: 5
   Index type: Flat Inner Product (cosine similarity)
   Search: exact nearest neighbor — no approximation

📊 VRAM used so far: 1.35 GB / 40.0 GB


## Cell 6 — Load Biomedical NER Model

In [7]:
from transformers import pipeline, AutoTokenizer, AutoModelForTokenClassification

print('Loading biomedical NER model...')
NER_MODEL = 'd4data/biomedical-ner-all'
ner_tokenizer = AutoTokenizer.from_pretrained(NER_MODEL)
ner_model_obj = AutoModelForTokenClassification.from_pretrained(NER_MODEL).to(DEVICE)

ner_pipe = pipeline(
    'ner',
    model=ner_model_obj,
    tokenizer=ner_tokenizer,
    aggregation_strategy='simple',
    device=0
)

print('✅ Biomedical NER model loaded')
print(f'   VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB')

Loading biomedical NER model...


tokenizer_config.json:   0%|          | 0.00/373 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1617: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/266M [00:00<?, ?B/s]

✅ Biomedical NER model loaded
   VRAM used: 1.62 GB


## Cell 7 — Load OpenBioLLM-8B

`aaditya/Llama3-OpenBioLLM-8B` — LLaMA 3 8B fine-tuned on curated clinical datasets.
Loaded in bfloat16 on A100 (no quantization needed — plenty of VRAM).

In [8]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

LLM_MODEL = 'aaditya/Llama3-OpenBioLLM-8B'
print(f'Loading {LLM_MODEL}...')

llm_tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)
llm_tokenizer.pad_token = llm_tokenizer.eos_token

if LOAD_IN_4BIT:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.bfloat16
    )
    llm_model = AutoModelForCausalLM.from_pretrained(
        LLM_MODEL, quantization_config=bnb_config, device_map='auto'
    )
else:
    llm_model = AutoModelForCausalLM.from_pretrained(
        LLM_MODEL, torch_dtype=torch.bfloat16, device_map='auto'
    )

print(f'✅ OpenBioLLM-8B loaded')
print(f'   VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB / 40.0 GB')

Loading aaditya/Llama3-OpenBioLLM-8B...


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/449 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/704 [00:00<?, ?B/s]

...

pytorch_model-00003-of-00004.bin:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

pytorch_model-00004-of-00004.bin:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

✅ OpenBioLLM-8B loaded
   VRAM used: 17.68 GB / 40.0 GB


## Cell 8 — Retrieval + NER Functions

Core RAG retrieval: embed query → search FAISS → return top-k chunks.
NER post-processing: clean subword tokens (fix the `##` artifact from previous version).

In [9]:
import pandas as pd
from collections import defaultdict

LABEL_MAP = {
    'Disease_disorder':      '🔴 Diseases & Disorders',
    'Medication':            '💊 Medications',
    'Sign_symptom':          '🟡 Signs & Symptoms',
    'Anatomical_location':   '🫀 Anatomy',
    'Lab_value':             '🧪 Lab Values',
    'Diagnostic_procedure':  '📋 Diagnostic Procedures',
    'Therapeutic_procedure': '💉 Therapeutic Procedures',
    'Biological_structure':  '🔬 Biological Structures',
    'Clinical_event':        '📌 Clinical Events',
    'Age':                   '👤 Demographics',
    'Sex':                   '👤 Demographics',
    'Dosage':                '💊 Dosages',
    'Duration':              '⏱ Duration',
}

# ── RETRIEVAL ──────────────────────────────────────────────────────────
def retrieve_chunks(query: str, top_k: int = 4) -> List[Dict]:
    """
    Embed query with BGE prefix + search FAISS index.
    BGE models use 'Represent this sentence for searching relevant passages: '
    prefix on the query side for best retrieval performance.
    """
    prefixed_query = f'Represent this sentence for searching relevant passages: {query}'
    query_emb = embedder.encode(
        [prefixed_query],
        normalize_embeddings=True
    ).astype(np.float32)

    scores, indices = faiss_index_gpu.search(query_emb, top_k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx != -1:
            chunk = all_chunks[idx].copy()
            chunk['similarity_score'] = float(score)
            results.append(chunk)

    return results


# ── NER WITH ## FIX ────────────────────────────────────────────────────
def run_ner_clean(text: str) -> Dict:
    """
    Run biomedical NER with subword token cleanup.
    Filters out any entity containing ## artifacts from WordPiece tokenization.
    """
    raw = ner_pipe(text.strip())
    grouped = defaultdict(list)
    seen = set()

    for ent in raw:
        word = ent['word'].strip()
        label = ent['entity_group']
        score = round(ent['score'], 3)

        # Fix 1: Skip subword artifacts
        if '##' in word:
            continue

        # Fix 2: Skip very short fragments
        if len(word) < 3:
            continue

        # Fix 3: Skip low confidence
        if score < 0.6:
            continue

        # Deduplicate
        key = (word.lower(), label)
        if key not in seen:
            seen.add(key)
            grouped[label].append({'entity': word, 'confidence': score})

    return grouped


def display_ner_results(grouped: Dict) -> pd.DataFrame:
    """Pretty print NER results and return as DataFrame."""
    rows = []
    for label, ents in grouped.items():
        readable = LABEL_MAP.get(label, label)
        for e in ents:
            rows.append({
                'Category': readable,
                'Entity': e['entity'],
                'Confidence': e['confidence']
            })
    return pd.DataFrame(rows)


print('✅ Retrieval and NER functions ready')
print('   retrieve_chunks(query, top_k) — semantic search over patient record')
print('   run_ner_clean(text) — NER with ## artifact fix')

✅ Retrieval and NER functions ready
   retrieve_chunks(query, top_k) — semantic search over patient record
   run_ner_clean(text) — NER with ## artifact fix


## Cell 9 — Full RAG Generation Pipeline

Query → Retrieve relevant chunks → Run NER → Feed to OpenBioLLM-8B → Structured output.

In [10]:
def generate_rag_response(
    query: str,
    top_k: int = 4,
    max_new_tokens: int = 600,
    verbose: bool = True
) -> Dict:
    """
    Full RAG pipeline:
    1. Retrieve top-k relevant chunks from FAISS
    2. Run NER on retrieved chunks
    3. Build grounded prompt with retrieved context
    4. Generate response with OpenBioLLM-8B
    """
    if verbose:
        print(f'\n🔍 Query: {query}')
        print('='*60)

    # Step 1: Retrieve
    retrieved = retrieve_chunks(query, top_k=top_k)
    if verbose:
        print(f'\n📚 Retrieved {len(retrieved)} chunks:')
        for i, r in enumerate(retrieved):
            print(f'   [{i+1}] {r["doc"]} | Score: {r["similarity_score"]:.4f}')
            print(f'       {r["text"][:100]}...')

    # Step 2: NER on retrieved context
    combined_context = ' '.join([r['text'] for r in retrieved])
    ner_results = run_ner_clean(combined_context)

    entity_lines = []
    for label, ents in ner_results.items():
        readable = LABEL_MAP.get(label, label)
        terms = ', '.join([e['entity'] for e in ents])
        if terms:
            entity_lines.append(f'- {readable}: {terms}')
    entity_summary = '\n'.join(entity_lines)

    if verbose:
        print(f'\n🏷️  NER extracted {sum(len(v) for v in ner_results.values())} entities')

    # Step 3: Build RAG prompt
    context_text = '\n\n'.join([
        f'[Source: {r["doc"]}]\n{r["text"]}'
        for r in retrieved
    ])

    prompt = (
        '<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n'
        'You are a clinical AI assistant. Answer questions about patients using ONLY '
        'the provided clinical context. Be precise, structured, and cite the source document.\n'
        '<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n'
        f'CLINICAL CONTEXT (retrieved from patient record):\n{context_text}\n\n'
        f'EXTRACTED CLINICAL ENTITIES (from NER):\n{entity_summary}\n\n'
        f'QUERY: {query}\n\n'
        'Provide a structured, grounded response using only the information above.'
        '<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n'
    )

    # Step 4: Generate
    inputs = llm_tokenizer(
        prompt,
        return_tensors='pt',
        truncation=True,
        max_length=3072
    ).to(llm_model.device)

    with torch.no_grad():
        out = llm_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.2,
            top_p=0.9,
            do_sample=True,
            pad_token_id=llm_tokenizer.eos_token_id,
            repetition_penalty=1.1
        )

    gen_tokens = out[0][inputs['input_ids'].shape[1]:]
    response = llm_tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()

    if verbose:
        print(f'\n🤖 OpenBioLLM-8B Response:')
        print('='*60)
        print(response)

    return {
        'query': query,
        'retrieved_chunks': retrieved,
        'ner_results': ner_results,
        'response': response
    }


print('✅ RAG pipeline ready')
print('   Usage: generate_rag_response(query, top_k=4)')

✅ RAG pipeline ready
   Usage: generate_rag_response(query, top_k=4)


## Cell 10 — Generate Full SOAP Note from Patient Record

Run the full RAG pipeline to generate a comprehensive SOAP note
grounded in the entire patient record.

In [11]:
soap_result = generate_rag_response(
    query=(
        'Generate a comprehensive SOAP note for this patient. '
        'Include: S (Subjective complaints), O (Objective findings, vitals, labs), '
        'A (Assessment with all diagnoses and ICD-10 codes), '
        'P (Plan with all medications, referrals, follow-up). '
        'Also list Key Clinical Concerns and current medication list.'
    ),
    top_k=5,
    max_new_tokens=700,
    verbose=True
)

# Display NER entity table
print('\n📊 Extracted Entity Table:')
entity_df = display_ner_results(soap_result['ner_results'])
print(entity_df.to_string(index=False))


🔍 Query: Generate a comprehensive SOAP note for this patient. Include: S (Subjective complaints), O (Objective findings, vitals, labs), A (Assessment with all diagnoses and ICD-10 codes), P (Plan with all medications, referrals, follow-up). Also list Key Clinical Concerns and current medication list.

📚 Retrieved 5 chunks:
   [1] Radiology Report - Chest XRay | Score: 0.5818
       Patient: John M., DOB: 03/15/1966. Date: March 2, 2024. Examination: PA and lateral chest radiograph...
   [2] Visit 3 - Endocrinology Consult | Score: 0.5600
       Patient: John M., 58-year-old male. Date: April 18, 2024. Referring: Primary Care. Reason for referr...
   [3] Lab Report - Comprehensive Panel | Score: 0.5591
       Patient: John M. Date: April 18, 2024. Ordered by: Endocrinology. METABOLIC PANEL:
    Glucose: 174 ...
   [4] Visit 1 - Initial Presentation | Score: 0.5550
       Patient: John M., 58-year-old male. Date: January 15, 2024. Chief Complaint: Increased thirst, frequ...
   [5] Visit

Starting from v4.46, the `logits` model output will have the same type as the model (except at train time, where it will always be FP32)



🤖 OpenBioLLM-8B Response:
The Answer is a comprehensive SOAP note for the patient.

📊 Extracted Entity Table:
               Category               Entity  Confidence
   Detailed_description              lateral       0.911
   Detailed_description                 dual       0.997
🔬 Biological Structures                chest       0.999
🔬 Biological Structures            pulmonary       0.995
📋 Diagnostic Procedures                radio       1.000
📋 Diagnostic Procedures           heart size       0.998
📋 Diagnostic Procedures cardiothoracic ratio       0.999
📋 Diagnostic Procedures      fasting glucose       0.983
📋 Diagnostic Procedures              peptide       0.663
📋 Diagnostic Procedures              insulin       0.976
📋 Diagnostic Procedures   glucose monitoring       0.996
 🔴 Diseases & Disorders             diabetes       0.987
     🟡 Signs & Symptoms             enlarged       0.995
     🟡 Signs & Symptoms  vascular congestion       0.898
     🟡 Signs & Symptoms         ca

## Cell 11 — Run Specific Clinical Queries

This demonstrates the RAG retrieval — each query retrieves different relevant chunks
from the patient record, grounding the LLM answer in the right source documents.

In [12]:
QUERIES = [
    'What medications is the patient currently taking and what are the doses?',
    'What are the patient lab values and which ones are abnormal?',
    'What kidney function findings are documented and what is the staging?',
    'What is the radiology report finding and clinical impression?',
]

query_results = []

for query in QUERIES:
    print('\n' + '='*70)
    result = generate_rag_response(query, top_k=3, max_new_tokens=350, verbose=True)
    query_results.append(result)
    print()



🔍 Query: What medications is the patient currently taking and what are the doses?

📚 Retrieved 3 chunks:
   [1] Visit 3 - Endocrinology Consult | Score: 0.6484
       Patient: John M., 58-year-old male. Date: April 18, 2024. Referring: Primary Care. Reason for referr...
   [2] Lab Report - Comprehensive Panel | Score: 0.6081
       Patient: John M. Date: April 18, 2024. Ordered by: Endocrinology. METABOLIC PANEL:
    Glucose: 174 ...
   [3] Visit 2 - Follow-up | Score: 0.6047
       Patient: John M., 58-year-old male. Date: March 2, 2024. Interval History: Patient reports improved ...

🏷️  NER extracted 38 entities

🤖 OpenBioLLM-8B Response:
The patient is currently taking the following medications: - Metformin at a dosage of 1000mg twice daily. - Empagliflozin at a dosage of 10mg daily. - Lisinopril at a dosage of 10mg daily. - Gabapentin at a dosage of 300mg nightly.



🔍 Query: What are the patient lab values and which ones are abnormal?

📚 Retrieved 3 chunks:
   [1] Lab Report - 

## Cell 12 — Ask Your Own Question

In [13]:
# Ask anything about the patient record
YOUR_QUERY = 'What is the progression of HbA1c across visits and is glycemic control improving?'

custom_result = generate_rag_response(
    YOUR_QUERY,
    top_k=4,
    max_new_tokens=400,
    verbose=True
)


🔍 Query: What is the progression of HbA1c across visits and is glycemic control improving?

📚 Retrieved 4 chunks:
   [1] Visit 3 - Endocrinology Consult | Score: 0.6784
       Patient: John M., 58-year-old male. Date: April 18, 2024. Referring: Primary Care. Reason for referr...
   [2] Visit 1 - Initial Presentation | Score: 0.6300
       Patient: John M., 58-year-old male. Date: January 15, 2024. Chief Complaint: Increased thirst, frequ...
   [3] Visit 2 - Follow-up | Score: 0.6213
       Patient: John M., 58-year-old male. Date: March 2, 2024. Interval History: Patient reports improved ...
   [4] Lab Report - Comprehensive Panel | Score: 0.6053
       Patient: John M. Date: April 18, 2024. Ordered by: Endocrinology. METABOLIC PANEL:
    Glucose: 174 ...

🏷️  NER extracted 53 entities

🤖 OpenBioLLM-8B Response:
The progression of HbA1c across visits is as follows:  Visit 1 - January 15, 2024: HbA1c 9.4% Visit 2 - March 2, 2024: HbA1c 8.6% Visit 3 - April 18, 2024: HbA1c 8.1%  Based o

## Cell 13 — Pipeline Performance Summary

In [14]:
print('='*60)
print('  CLINICAL NLP RAG PIPELINE — SUMMARY')
print('='*60)

print(f'\n📄 Patient Record:')
print(f'   Documents: {len(PATIENT_RECORD)}')
print(f'   Total chunks: {len(all_chunks)}')
print(f'   Total words: {sum(len(t.split()) for t in PATIENT_RECORD.values())}')

print(f'\n🔧 Models Used:')
print(f'   Embeddings: BAAI/bge-large-en-v1.5 (1024-dim)')
print(f'   Vector DB:  FAISS GPU (Flat IP, exact search)')
print(f'   NER:        d4data/biomedical-ner-all')
print(f'   LLM:        aaditya/Llama3-OpenBioLLM-8B')

print(f'\n📊 GPU Memory:')
allocated = torch.cuda.memory_allocated() / 1e9
total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'   Used: {allocated:.2f} GB / {total:.1f} GB')
print(f'   Free: {total - allocated:.2f} GB')

print(f'\n✅ Pipeline Components:')
print(f'   [1] Semantic chunking with 64-token overlap')
print(f'   [2] BGE-Large biomedical embeddings')
print(f'   [3] FAISS GPU vector index')
print(f'   [4] Biomedical NER with ## artifact fix')
print(f'   [5] RAG-grounded OpenBioLLM-8B generation')

print(f'\n🏥 Cotiviti Relevance:')
print(f'   → Automates clinical coding from unstructured notes')
print(f'   → RAG grounds LLM in actual patient documents')
print(f'   → Scales to full claim packages (hundreds of pages)')
print(f'   → Directly supports payment integrity workflows')

  CLINICAL NLP RAG PIPELINE — SUMMARY

📄 Patient Record:
   Documents: 5
   Total chunks: 5
   Total words: 568

🔧 Models Used:
   Embeddings: BAAI/bge-large-en-v1.5 (1024-dim)
   Vector DB:  FAISS GPU (Flat IP, exact search)
   NER:        d4data/biomedical-ner-all
   LLM:        aaditya/Llama3-OpenBioLLM-8B

📊 GPU Memory:
   Used: 17.68 GB / 42.4 GB
   Free: 24.73 GB

✅ Pipeline Components:
   [1] Semantic chunking with 64-token overlap
   [2] BGE-Large biomedical embeddings
   [3] FAISS GPU vector index
   [4] Biomedical NER with ## artifact fix
   [5] RAG-grounded OpenBioLLM-8B generation

🏥 Cotiviti Relevance:
   → Automates clinical coding from unstructured notes
   → RAG grounds LLM in actual patient documents
   → Scales to full claim packages (hundreds of pages)
   → Directly supports payment integrity workflows


## Cell 14 — Save All Outputs

In [15]:
import os, json
os.makedirs('/content/outputs', exist_ok=True)

# Save chunks
chunks_df = pd.DataFrame(all_chunks)
chunks_df.to_csv('/content/outputs/chunks.csv', index=False)

# Save entity table from SOAP
entity_df.to_csv('/content/outputs/entities.csv', index=False)

# Save SOAP note
with open('/content/outputs/soap_note.txt', 'w') as f:
    f.write('SOAP NOTE — Generated by Clinical NLP RAG Pipeline\n')
    f.write('Model: OpenBioLLM-8B | RAG: FAISS + BGE-Large\n')
    f.write('='*60 + '\n\n')
    f.write(soap_result['response'])

# Save all query results
with open('/content/outputs/query_results.txt', 'w') as f:
    for r in query_results:
        f.write(f'QUERY: {r["query"]}\n')
        f.write('-'*50 + '\n')
        f.write(r['response'] + '\n\n')

print('✅ Outputs saved to /content/outputs/')
print('   chunks.csv        — all semantic chunks with metadata')
print('   entities.csv      — NER extracted entities')
print('   soap_note.txt     — full RAG-generated SOAP note')
print('   query_results.txt — all clinical query responses')
print('\nDownload via Files panel on the left sidebar.')

✅ Outputs saved to /content/outputs/
   chunks.csv        — all semantic chunks with metadata
   entities.csv      — NER extracted entities
   soap_note.txt     — full RAG-generated SOAP note
   query_results.txt — all clinical query responses

Download via Files panel on the left sidebar.


---

## Pipeline Summary

| Component | Choice | Purpose |
|---|---|---|
| Chunking | Semantic + 64-token overlap | Preserve context at boundaries |
| Embeddings | `BAAI/bge-large-en-v1.5` | 1024-dim dense biomedical embeddings |
| Vector DB | FAISS GPU (Flat IP) | Exact cosine similarity search |
| NER | `d4data/biomedical-ner-all` | Extract clinical entities |
| LLM | `OpenBioLLM-8B` | RAG-grounded clinical generation |

**Why RAG over plain generation:**
- Grounds LLM responses in actual patient documents — no hallucination
- Scales to full clinical record (hundreds of pages) without context window limits
- Each answer is traceable to a source document
- Directly mirrors Cotiviti's real-world use case: payer claim package review

**Author:** Arshad Naguru | MS AI, Rochester Institute of Technology | an2629@rit.edu